In [18]:
import pandas as pd
import numpy as np

# ── Step 1: Load data ──────────────────────────────────────────────────────
df = pd.read_csv("../Dataset/original_data.csv", header=None, names=range(26), dtype=str)
print("Original shape:", data.shape)
print("\nFirst 5 rows:")
print(data.head())

Original shape: (2509, 26)

First 5 rows:
              0         1         2         3         4         5         6   \
0           Year      2017       NaN       NaN       NaN       NaN       NaN   
1   January-2017  00:00:00  01:00:00  02:00:00  03:00:00  04:00:00  05:00:00   
2             \n       NaN       NaN       NaN       NaN       NaN       NaN   
3  February-2017  00:00:00  01:00:00  02:00:00  03:00:00  04:00:00  05:00:00   
4             \n       NaN       NaN       NaN       NaN       NaN       NaN   

         7         8         9   ...        16        17        18        19  \
0       NaN       NaN       NaN  ...       NaN       NaN       NaN       NaN   
1  06:00:00  07:00:00  08:00:00  ...  15:00:00  16:00:00  17:00:00  18:00:00   
2       NaN       NaN       NaN  ...       NaN       NaN       NaN       NaN   
3  06:00:00  07:00:00  08:00:00  ...  15:00:00  16:00:00  17:00:00  18:00:00   
4       NaN       NaN       NaN  ...       NaN       NaN       NaN       NaN 

In [19]:
import pandas as pd
import numpy as np

# STEP 1: Load data and force 26 columns so it doesn't crash
print("Loading data...")
df = pd.read_csv("../Dataset/original_data.csv", header=None, names=range(26), dtype=str)

# STEP 2: Clean up random hidden characters (like \n) in the first column
df[0] = df[0].str.strip()

# STEP 3: Drop completely empty rows or rows that just became empty
df = df.dropna(how='all')
df = df[df[0] != '']

# STEP 4: Remove the "Year" rows
df = df[df[0] != 'Year']

# STEP 5: Handle the Dates (Forward Fill the Month-Year)
# Your logic: If the cell has a hyphen (like 'January-2017'), it's our month label!
df['Month_Year'] = np.where(df[0].str.contains('-', na=False), df[0], np.nan)
df['Month_Year'] = df['Month_Year'].ffill() # Forward-fill copies it downwards

# STEP 6: Keep only the Day rows
# Now we just filter to keep rows where column 0 is a number (1, 2, 3...)
df = df[df[0].str.isnumeric() == True]

# STEP 7: Prepare for Melting
# Drop column 25 as it is completely empty (NaN) based on the screenshot
df = df.drop(columns=[25])

# Rename Column 0 to 'Day'
df = df.rename(columns={0: 'Day'})

# Rename Columns 1 through 24 to proper hour times ('00:00:00', '01:00:00', etc.)
hours = [f"{str(i).zfill(2)}:00:00" for i in range(24)]
hour_mapping = {i+1: hours[i] for i in range(24)}
df = df.rename(columns=hour_mapping)

# Reorder columns so 'Month_Year' is first and 'Day' is second
df = df[['Month_Year', 'Day'] + hours]

# STEP 8: MELT! (The Unpivot)
print("Melting data...")
df_melted = pd.melt(
    df, 
    id_vars=['Month_Year', 'Day'], # Keep dates as they are
    value_vars=hours,              # Melt the time given in a single row
    var_name='Time', 
    value_name='AQI'               # Keep the values too
)

# STEP 9: Final cleanup (Datetime and Numeric conversion)
# %B handles full month names like "January", %Y handles 4-digit years
df_melted['Datetime'] = pd.to_datetime(
    df_melted['Month_Year'] + '-' + df_melted['Day'] + ' ' + df_melted['Time'], 
    format='%B-%Y-%d %H:%M:%S',
    errors='coerce'
)

# Drop invalid dates (like Feb 30th) and sort chronologically
df_melted = df_melted.dropna(subset=['Datetime']).sort_values('Datetime').reset_index(drop=True)

# Convert AQI to numbers so it's ready for Machine Learning
df_melted['AQI'] = pd.to_numeric(df_melted['AQI'], errors='coerce')

# Reorder the final melted dataset to be super clean
df_melted = df_melted[['Datetime', 'Month_Year', 'Day', 'Time', 'AQI']]

print("Success! Final Shape:", df_melted.shape)
print(df_melted.head())
print(df_melted.tail())

Loading data...
Melting data...
Success! Final Shape: (55584, 5)
             Datetime Month_Year Day      Time  AQI
0 2017-07-01 00:00:00  July-2017   1  00:00:00  NaN
1 2017-07-01 01:00:00  July-2017   1  01:00:00  NaN
2 2017-07-01 02:00:00  July-2017   1  02:00:00  NaN
3 2017-07-01 03:00:00  July-2017   1  03:00:00  NaN
4 2017-07-01 04:00:00  July-2017   1  04:00:00  NaN
                 Datetime     Month_Year Day      Time    AQI
55579 2023-12-31 19:00:00  December-2023  31  19:00:00  385.0
55580 2023-12-31 20:00:00  December-2023  31  20:00:00  375.0
55581 2023-12-31 21:00:00  December-2023  31  21:00:00  350.0
55582 2023-12-31 22:00:00  December-2023  31  22:00:00  336.0
55583 2023-12-31 23:00:00  December-2023  31  23:00:00  333.0


In [20]:
df_melted.tail()

,Datetime,Month_Year,Day,Time,AQI
55579,2023-12-31 19:00:00,December-2023,31,19:00:00,385.0
55580,2023-12-31 20:00:00,December-2023,31,20:00:00,375.0
55581,2023-12-31 21:00:00,December-2023,31,21:00:00,350.0
55582,2023-12-31 22:00:00,December-2023,31,22:00:00,336.0
55583,2023-12-31 23:00:00,December-2023,31,23:00:00,333.0


In [21]:
# STEP 10: Save the cleaned data to a new file
print("\nSaving cleaned data to CSV...")
# index=False prevents pandas from saving the row numbers (0, 1, 2...) as a new column
df_melted.to_csv("../Dataset/cleaned_data.csv", index=False)
print("Data successfully saved to '../Dataset/cleaned_data.csv'!")


Saving cleaned data to CSV...
Data successfully saved to '../Dataset/cleaned_data.csv'!
